In [16]:
import sys
import re

sys.path.insert(0, '..')

import pandas as pd
from utils import classify_sentiment, normalize_text

df = pd.read_csv('../data/ai_apps_reviews_100000.csv')

if 'Sentiment' not in df.columns:
    df['Sentiment'] = df['score'].apply(classify_sentiment)

topic_df = df.dropna(subset=['appName', 'content', 'score', 'Sentiment']).copy()
topic_df['content'] = topic_df['content'].apply(normalize_text)
topic_df = topic_df[topic_df['content'] != ''].reset_index(drop=True)

print('Rows:', len(topic_df))
print('Sentiment counts:', topic_df['Sentiment'].value_counts().to_dict())

Rows: 233328
Sentiment counts: {'Positive': 194408, 'Negative': 28620, 'Neutral': 10300}


In [17]:
# Topic modeling (LDA) with Gensim

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from gensim import corpora
from gensim.models import LdaModel



def simple_tokenize(text: str):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    toks = [t for t in text.split() if len(t) >= 3 and t not in ENGLISH_STOP_WORDS]
    return toks

for sent in ['Negative', 'Neutral', 'Positive']:
    subset = topic_df[topic_df['Sentiment'] == sent]
    subset = subset.sample(n=min(1500, len(subset)), random_state=42)

    texts = [simple_tokenize(t) for t in subset['content'].tolist()]
    texts = [t for t in texts if len(t) >= 3]

    dictionary = corpora.Dictionary(texts)
    dictionary.filter_extremes(no_below=5, no_above=0.5)
    corpus = [dictionary.doc2bow(t) for t in texts]

    if len(dictionary) < 50 or len(corpus) < 50:
        print(f'Not enough data for LDA on {sent} after filtering.')
        continue

    lda = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=6,
        random_state=42,
        passes=8,
        alpha='auto',
        per_word_topics=False,
    )

    print(f"\nTop topics for {sent} reviews")
    for i in range(6):
        print(f"Topic {i}:", lda.print_topic(i, topn=10))


Top topics for Negative reviews
Topic 0: 0.035*"app" + 0.028*"image" + 0.022*"like" + 0.021*"phone" + 0.020*"images" + 0.020*"free" + 0.019*"help" + 0.018*"error" + 0.017*"create" + 0.016*"edit"
Topic 1: 0.029*"bad" + 0.025*"work" + 0.023*"time" + 0.019*"gemini" + 0.017*"day" + 0.016*"app" + 0.016*"long" + 0.014*"money" + 0.014*"slow" + 0.013*"images"
Topic 2: 0.071*"app" + 0.055*"number" + 0.051*"phone" + 0.020*"hai" + 0.019*"invalid" + 0.018*"don" + 0.018*"need" + 0.017*"account" + 0.014*"gemini" + 0.014*"information"
Topic 3: 0.058*"chat" + 0.026*"just" + 0.025*"gpt" + 0.021*"photo" + 0.019*"app" + 0.019*"don" + 0.017*"apps" + 0.016*"chats" + 0.016*"image" + 0.015*"want"
Topic 4: 0.038*"app" + 0.026*"chatgpt" + 0.025*"claude" + 0.023*"like" + 0.023*"use" + 0.020*"version" + 0.019*"good" + 0.017*"free" + 0.017*"better" + 0.015*"don"
Topic 5: 0.033*"google" + 0.032*"chat" + 0.023*"voice" + 0.022*"time" + 0.017*"use" + 0.015*"good" + 0.014*"subscription" + 0.013*"assistant" + 0.013*"n

In [18]:
# Named Entity Recognition (spaCy)

import spacy
from collections import Counter

nlp = spacy.load('en_core_web_sm')

sample_n = min(2000, len(topic_df))
sample = topic_df.sample(n=sample_n, random_state=42).copy()

entity_counts_by_sent = {s: Counter() for s in ['Negative', 'Neutral', 'Positive']}

for sent, text in zip(sample['Sentiment'].tolist(), sample['content'].tolist()):
    doc = nlp(text)
    for ent in doc.ents:
        key = (ent.text.strip().lower(), ent.label_)
        if key[0]:
            entity_counts_by_sent[sent][key] += 1

print('Top entities by sentiment (sample)')
for sent in ['Negative', 'Neutral', 'Positive']:
    print(f"\n{sent}")
    for (txt, label), cnt in entity_counts_by_sent[sent].most_common(15):
        print(f"  {cnt:>4}  {label:<8}  {txt}")

Top entities by sentiment (sample)

Negative
     9  CARDINAL  one
     8  GPE       ai
     8  ORG       google
     6  PERSON    claude
     5  GPE       gemini
     5  ORDINAL   first
     3  CARDINAL  1
     3  PERSON    grok
     3  ORG       gpt
     3  ORG       ai
     3  ORG       api
     3  PERSON    gemini
     3  GPE       hai
     3  ORDINAL   third
     3  PERSON    david

Neutral
     2  ORG       google
     1  TIME      hourly
     1  ORDINAL   5hrs
     1  ORDINAL   1hr
     1  ORG       gemini
     1  ORDINAL   first
     1  DATE      the last couple weeks
     1  TIME      a couple hours
     1  PERCENT   over 50%
     1  PERSON    dan
     1  GPE       gemini
     1  TIME      hours or minutes
     1  ORG       chatgpt

Positive
    27  GPE       ai
    27  PERSON    claude
    20  CARDINAL  one
    19  GPE       hai
    17  ORG       gpt
    14  GPE       gemini
    10  PERSON    ️‍🩹
     9  ORG       claude
     8  ORG       chatgpt
     7  CARDINAL  5
     7  O